# Session 3 - Analysis and Statistics

Goal: parse model outputs, quantify syntactic priming, and visualize results.

## What's New in This Expanded Session

| Section | What You'll Learn |
|---------|-------------------|
| 1. Three Parser Methods | Rule-based vs spaCy dependency vs LLM-as-judge |
| 2. Parser Validation | Precision, recall, F1 against gold standard |
| 3. Rich Metrics | Priming rate, effect size, bootstrap CI, response length, lexical diversity, sentiment |
| 4. Visualization | Bar charts, box plots, heatmaps, scatter plots |
| 5. Multi-Condition Comparison | Tukey HSD post-hoc test |
| 6. Advanced Statistics | Chi-square, logistic regression, mixed-effects, power analysis |
| 7. A/B Testing | Compare baseline vs optimized prompt strategies |
| 8. Summary & Next Steps | What you've learned and what's next |

## Learning Objectives

1. Convert raw LLM responses into analysis-ready features using multiple parsing strategies.
2. Validate parser quality against hand-labeled data.
3. Compute priming rates, effect sizes, and confidence intervals.
4. Create publication-quality visualizations.
5. Run chi-square, logistic regression, and mixed-effects models.
6. Understand statistical power and experimental design.
7. Run A/B tests comparing prompt strategies.

## TODO Mapping to Source Files

After this notebook, complete these modules:

| File | Role |
|---|---|
| `src/analysis/parser.py` | Rule-based, dependency, and LLM-judge parsing |
| `src/analysis/metrics.py` | Priming rate, effect size, bootstrap CI, lexical diversity, sentiment |
| `src/analysis/stats.py` | Chi-square, logistic regression, mixed-effects, power analysis, Tukey HSD |
| `src/analysis/visualizer.py` | Bar charts, box plots, heatmaps, scatter plots |
| `src/optimization/ab_test.py` | A/B testing helpers |

In [ ]:
# Path setup
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
SRC_ROOT = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_ROOT))

print("Project root:", PROJECT_ROOT)
print("Source root:", SRC_ROOT)

---
## 0. Load Session 2 Output

Load the trial outputs generated by Session 2's `execute_experiment()`.
These are stored as JSONL files in `artifacts/runs/`.

**Update the path below** to point to your actual Session 2 run output file.

In [ ]:
import json
from pathlib import Path

# --- CONFIG: Update this path to your Session 2 output file ---
RUN_OUTPUT_PATH = Path("../artifacts/runs/run_001_seed42.jsonl")
# -------------------------------------------------------------

def load_trial_outputs(path: Path) -> list[dict]:
    """Load trial outputs from a JSONL file."""
    if not path.exists():
        print(f"[WARNING] File not found: {path}")
        print("Please update RUN_OUTPUT_PATH to point to your Session 2 output file.")
        return []
    outputs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                outputs.append(json.loads(line))
    return outputs

trial_outputs = load_trial_outputs(RUN_OUTPUT_PATH)
print(f"Loaded {len(trial_outputs)} trial outputs from {RUN_OUTPUT_PATH}")

if trial_outputs:
    CONDITIONS = sorted(set(o["metadata"]["condition"] for o in trial_outputs))
    print(f"Conditions found: {CONDITIONS}")
    print(f"\nFirst record:")
    print(json.dumps(trial_outputs[0], indent=2))

---
## 1. Three Parser Methods

We need to detect what syntactic structure the LLM used in its response.
There are three approaches, each with different trade-offs:

| Method | Approach | Pros | Cons |
|--------|----------|------|------|
| **Rule-based** | Regex patterns for active/passive voice | Fast, no dependencies | Brittle, misses edge cases |
| **Dependency** | spaCy dependency tree (nsubj vs nsubjpass) | Linguistically grounded | Requires model download |
| **LLM-as-judge** | Ask an LLM to classify the sentence | Flexible, handles ambiguity | Slow, costs tokens |

**Your task**: Implement `parse_trial_output()`, `parse_with_dependency()`, and `parse_with_llm_judge()` in `src/analysis/parser.py`.

**Exercise**: Compare all three on the same data and measure agreement.

In [ ]:
# --- 1a. Rule-Based Parser ---
# Implement this in src/analysis/parser.py -> parse_trial_output()
#
# Hint: Use regex patterns to detect active vs passive voice.
# Active example: "The chef praised the assistant."
# Passive example: "The assistant was praised by the chef."
#
# Your function should return a dict with keys:
#   stimulus_id, condition, used_target_structure (bool),
#   detected_structure, parser_method, parser_confidence,
#   response_text, response_length

import re

# TODO: Define ACTIVE_PATTERNS and PASSIVE_PATTERNS regex lists
# TODO: Implement rule_based_classify(text) -> (structure, confidence)
# TODO: Implement parse_rule_based(trial_output) -> dict

# Test your parser on a few examples:
test_sentences = [
    "The chef praised the assistant.",
    "The assistant was praised by the chef.",
    "There was a chef in the kitchen.",
]
for s in test_sentences:
    # struct, conf = rule_based_classify(s)
    # print(f"  {struct:8s} (conf={conf:.2f}) | {s}")
    pass

In [ ]:
# --- 1b. Dependency Parser (spaCy) ---
# Implement this in src/analysis/parser.py -> parse_with_dependency()
# Requires: pip install spacy && python -m spacy download en_core_web_sm
#
# Hint: Use spaCy's dependency labels:
#   - 'nsubj' = nominal subject (active voice)
#   - 'nsubjpass' = passive subject
#   - 'auxpass' = passive auxiliary
#
# If spaCy is not available, fall back to rule-based.

# TODO: Implement parse_with_dependency_local(trial_output) -> dict

# Test on first record:
# result = parse_with_dependency_local(trial_outputs[0])
# print(result)

In [ ]:
# --- 1c. LLM-as-Judge Parser ---
# Implement this in src/analysis/parser.py -> parse_with_llm_judge()
#
# Hint: Ask an LLM: "Is this sentence active or passive? Respond with only 'active' or 'passive'."
#
# A mock judge is provided below for testing.
# In production, replace with a real LLM API call.

def mock_llm_judge(prompt: str) -> str:
    """Mock LLM judge for demonstration."""
    if "was" in prompt.lower() and "by" in prompt.lower():
        return "passive"
    return "active"

# TODO: Implement parse_with_llm_judge_local(trial_output, judge_func) -> dict

# Test:
# result = parse_with_llm_judge_local(trial_outputs[0], judge_func=mock_llm_judge)
# print(result)

In [ ]:
# --- 1d. Compare All Three Parsers ---
#
# After implementing all three parsers, compare their agreement.
# High agreement = parsers are reliable. Low agreement = need gold standard.

# TODO: Implement parse_batch(trial_outputs, method) -> list[dict]
# TODO: Compute pairwise agreement rates between parsers
# TODO: Show examples where parsers disagree

# parsed_rule = parse_batch(trial_outputs, "rule_based")
# parsed_dep = parse_batch(trial_outputs, "dependency")
# parsed_llm = parse_batch(trial_outputs, "llm_judge", judge_func=mock_llm_judge)
# print(f"Parsed {len(parsed_rule)} records with each method")
#
# def agreement_rate(list_a, list_b):
#     matches = sum(1 for a, b in zip(list_a, list_b) if a['used_target_structure'] == b['used_target_structure'])
#     return matches / len(list_a)
#
# print(f"Rule vs Dependency: {agreement_rate(parsed_rule, parsed_dep):.1%}")

---
## 2. Parser Validation

How do we know which parser is correct? We need a **gold standard** - hand-labeled data.

**Key terms**:
- **Precision**: Of the items the parser labeled as "target structure", how many were correct?
- **Recall**: Of the items that actually used the target structure, how many did the parser find?
- **F1 Score**: Harmonic mean of precision and recall (balance between them)
- **Confusion Matrix**: Table showing true positives, false positives, true negatives, false negatives

**Exercise**: Hand-label 10 responses, then compute precision, recall, and F1 for each parser.

In [ ]:
# --- 2a. Create Gold Standard Labels ---
#
# In practice, you would hand-label these. Here we simulate it.
# Take the first 10 records and manually determine the true structure.

# TODO: Create gold_standard list with hand-labeled 'used_target_structure' values
# gold_standard = []
# for i, to in enumerate(trial_outputs[:10]):
#     text = to["response_text"]
#     # Manually determine true structure...
#     gold_standard.append({
#         "stimulus_id": to["stimulus_id"],
#         "used_target_structure": ...,  # True or False
#     })

In [ ]:
# --- 2b. Compute Precision, Recall, F1 ---
# Implement this in src/analysis/parser.py -> validate_parser_accuracy()
#
# Hint: Compare parser output vs gold standard:
#   TP = parser said True, gold said True
#   FP = parser said True, gold said False
#   TN = parser said False, gold said False
#   FN = parser said False, gold said True
#
#   precision = TP / (TP + FP)
#   recall = TP / (TP + FN)
#   F1 = 2 * precision * recall / (precision + recall)

# TODO: Implement validate_parser_accuracy(parsed_records, gold_standard) -> dict
# TODO: Run validation for all three parsers and compare results

---
## 3. Rich Metrics

Now we compute meaningful metrics from the parsed data. Here's what each term means:

| Metric | What It Measures | Formula / Method |
|--------|-----------------|------------------|
| **Priming Rate** | % of responses that used the target syntactic structure | `count(used_target=True) / count(total)` per condition |
| **Effect Size** | How strong is the priming effect? | **Odds Ratio**: odds(treatment) / odds(control). **Cohen's d**: standardized difference between two rates |
| **Bootstrap CI** | How reliable is the estimate? | Resample data 1000x with replacement, compute rate each time, take 2.5th and 97.5th percentiles |
| **Response Length** | Do primed responses differ in length? | Mean, median, std of word count per condition |
| **Lexical Diversity** | Do primed responses use more varied vocabulary? | **Type-Token Ratio (TTR)** = unique_words / total_words. Higher = more diverse |
| **Sentiment** | Does priming affect response tone? | Polarity (-1 to +1) and subjectivity (0 to 1) using TextBlob |

**Your task**: Implement all metric functions in `src/analysis/metrics.py`.

In [ ]:
# --- 3a. Priming Rate ---
# Implement this in src/analysis/metrics.py -> compute_priming_rate()
#
# Hint: Group records by condition, count how many have used_target_structure=True
#
# from collections import Counter
# def compute_priming_rate(parsed_records):
#     condition_counts = Counter()
#     condition_success = Counter()
#     for rec in parsed_records:
#         cond = rec.get("condition")
#         condition_counts[cond] += 1
#         if rec.get("used_target_structure"):
#             condition_success[cond] += 1
#     return {cond: condition_success[cond]/condition_counts[cond] for cond in condition_counts}

# TODO: Implement and test compute_priming_rate()

In [ ]:
# --- 3b. Effect Size ---
# Implement this in src/analysis/metrics.py -> compute_effect_size()
#
# Odds Ratio = (success_t / fail_t) / (success_c / fail_c)
# Cohen's d = (rate_t - rate_c) / pooled_standard_error
#
# TODO: Implement compute_effect_size(parsed_records, treatment, control) -> dict
# TODO: Compare each primed condition against control

In [ ]:
# --- 3c. Bootstrap Confidence Intervals ---
# Implement this in src/analysis/metrics.py -> bootstrap_confidence_interval()
#
# Bootstrap = resample with replacement many times, compute metric each time.
# 95% CI = [2.5th percentile, 97.5th percentile] of the bootstrap distribution.
#
# import random
# def bootstrap_priming_rate(parsed_records, condition, num_samples=1000):
#     cond_records = [r for r in parsed_records if r['condition'] == condition]
#     n = len(cond_records)
#     boot_rates = []
#     for _ in range(num_samples):
#         sample = [random.choice(cond_records) for _ in range(n)]
#         rate = sum(1 for r in sample if r['used_target_structure']) / n
#         boot_rates.append(rate)
#     boot_rates.sort()
#     return (boot_rates[25], boot_rates[975])  # 95% CI

# TODO: Implement and compute bootstrap CIs for each condition

In [ ]:
# --- 3d. Response Length Analysis ---
# Implement this in src/analysis/metrics.py -> compute_response_length_stats()
#
# Hint: Use statistics.mean(), statistics.stdev(), statistics.median()
#
# TODO: Implement compute_response_length_stats(parsed_records) -> dict
# TODO: Print a table of mean, std, median, min, max length per condition

In [ ]:
# --- 3e. Lexical Diversity (Type-Token Ratio) ---
# Implement this in src/analysis/metrics.py -> compute_lexical_diversity()
#
# TTR = len(set(all_words)) / len(all_words)
# Higher TTR = more varied vocabulary.
#
# TODO: Implement compute_lexical_diversity(parsed_records) -> dict
# TODO: Compare TTR across conditions

In [ ]:
# --- 3f. Sentiment Analysis (Optional) ---
# Implement this in src/analysis/metrics.py -> compute_sentiment_scores()
# Requires: pip install textblob
#
# Polarity: -1 (negative) to +1 (positive)
# Subjectivity: 0 (objective) to 1 (subjective)
#
# from textblob import TextBlob
# blob = TextBlob("The chef praised the assistant.")
# print(blob.sentiment.polarity, blob.sentiment.subjectivity)
#
# TODO: Implement compute_sentiment_scores(parsed_records) -> dict

In [ ]:
# --- 3g. All Metrics Summary ---
# Implement this in src/analysis/metrics.py -> compute_all_metrics()
#
# TODO: Combine all metrics into one summary dict
# all_metrics = {
#     "priming_rates": ...,
#     "response_length": ...,
#     "lexical_diversity": ...,
#     "sentiment": ...,
# }

---
## 4. Visualization

Visualizations make patterns immediately visible. We'll create:

1. **Bar chart**: Priming rate by condition with error bars (bootstrap CI)
2. **Box plot**: Response length distribution by condition
3. **Heatmap**: Confusion matrix for parser validation
4. **Scatter plot**: Response length vs priming outcome (with jitter)
5. **Grouped bar chart**: Multi-metric comparison across conditions

**Your task**: Implement all visualization functions in `src/analysis/visualizer.py`.
Use `matplotlib` and `numpy` for plotting.

---
## 5. Multi-Condition Comparison

**Why not just run multiple t-tests?**

If you have 5 conditions, running t-tests on all pairs (10 comparisons) inflates the chance of false positives. This is called the **multiple comparison problem**.

**Solution: Tukey HSD (Honestly Significant Difference)**

Tukey HSD is a post-hoc test that adjusts p-values for multiple comparisons. It tells you which pairs of conditions are significantly different from each other, while controlling the overall error rate.

**Your task**: Implement `run_tukey_hsd()` in `src/analysis/stats.py`.
Requires: `pip install scipy statsmodels pandas`

```python
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import pandas as pd

df = pd.DataFrame([{"condition": r["condition"], "primed": int(r["used_target_structure"])} for r in parsed_records])
tukey = pairwise_tukeyhsd(endog=df['primed'], groups=df['condition'], alpha=0.05)
print(tukey)
```

---
## 6. Advanced Statistics

### 6a. Chi-Square Test

**What it does**: Tests whether there's a significant association between two categorical variables (condition and structure usage).

**How it works**:
1. Build a contingency table (conditions x [used_target, not_used])
2. Compare observed frequencies to expected frequencies (if no association)
3. If the difference is large enough, the association is statistically significant

**Cramer's V**: Effect size for chi-square (0 to 1, higher = stronger association)

```python
from scipy.stats import chi2_contingency
chi2, p, dof, expected = chi2_contingency(table)
```

### 6b. Logistic Regression

**What it does**: Models the probability of using target structure as a function of condition.
Unlike chi-square, it can handle multiple predictors and continuous variables.

**Interpretation**: Coefficients represent log-odds. A positive coefficient means that condition increases the odds of using the target structure.

### 6c. Mixed-Effects Model

**What it does**: Accounts for random effects (e.g., by stimulus_id).
Different stimuli may elicit different responses regardless of condition.
Without this, you might overestimate significance (pseudoreplication bias).

### 6d. Power Analysis

**What it does**: Answers the question "How many trials do I need?"

- **Effect size** (Cohen's d): 0.2 = small, 0.5 = medium, 0.8 = large
- **Power**: Probability of detecting an effect if it exists (target: 0.80)
- **Alpha**: Significance threshold (typically 0.05)

If your experiment is underpowered, you might miss real effects.

**Your task**: Implement all statistical functions in `src/analysis/stats.py`.

In [ ]:
# --- 6a. Chi-Square Test ---
# Implement this in src/analysis/stats.py -> run_chi_square_test()
#
# from scipy.stats import chi2_contingency
# import numpy as np
#
# Build contingency table:
# conditions = sorted(set(r['condition'] for r in parsed_records))
# table = []
# for cond in conditions:
#     n_yes = sum(1 for r in parsed_records if r['condition']==cond and r['used_target_structure'])
#     n_no = sum(1 for r in parsed_records if r['condition']==cond and not r['used_target_structure'])
#     table.append([n_yes, n_no])
#
# chi2, p, dof, expected = chi2_contingency(np.array(table))
#
# TODO: Implement run_chi_square_test() and print results

---
## 7. A/B Testing

**What is A/B Testing?**

A/B testing compares two versions of a prompt strategy (A = baseline, B = optimized) to see which performs better.

**Key metrics for comparison**:
- Priming rate difference (Delta)
- Effect size change
- Cost per trial (token usage)
- Statistical significance of improvement

**Your task**: Implement `run_ab_comparison()` and `recommend_rollout()` in `src/optimization/ab_test.py`.

```python
# Compare baseline and candidate summaries:
# delta = candidate_rate - baseline_rate
# relative_improvement = delta / baseline_rate
# Recommend rollout if improvement > minimum_effect threshold
```

In [ ]:
# --- 7. A/B Testing ---
# Implement this in src/optimization/ab_test.py
#
# TODO: Implement run_ab_comparison(baseline_summary, candidate_summary) -> dict
#   Should return: delta, relative_improvement, p_value, recommendation
#
# TODO: Implement recommend_rollout(ab_result, minimum_effect) -> dict
#   Should return: should_rollout (bool), rationale (str)
#
# Example:
# baseline = {"priming_rate": 0.65, "model": "gpt-4o-mini", "prompt_style": "minimal"}
# candidate = {"priming_rate": 0.78, "model": "gpt-4o-mini", "prompt_style": "instructed"}
# result = run_ab_comparison(baseline, candidate)
# decision = recommend_rollout(result, minimum_effect=0.05)
# print(decision)

---
## 8. Summary & Next Steps

### What You've Learned

| Skill | Applied |
|-------|---------|
| NLP Parsing | Rule-based, dependency, LLM-as-judge |
| Parser Validation | Precision, recall, F1, confusion matrix |
| Metrics | Priming rate, effect size, bootstrap CI |
| Text Analysis | Response length, lexical diversity, sentiment |
| Visualization | Bar charts, box plots, heatmaps, scatter plots |
| Statistics | Chi-square, logistic regression, mixed-effects, power analysis |
| A/B Testing | Compare baseline vs optimized prompt strategies |

### Next Steps -> Session 4

In Session 4, you will:
1. Design effective experiments with prompt variants
2. Use an LLM to **auto-generate a scientific report**
3. Discuss **priming effect and AI alignment** implications
4. Build a **final report** and **resume bullet points**

### TODO: Complete Source Files

After this notebook, implement the TODOs in:
- `src/analysis/parser.py` - all parsing functions
- `src/analysis/metrics.py` - all metric functions
- `src/analysis/stats.py` - all statistical test functions
- `src/analysis/visualizer.py` - all visualization functions
- `src/optimization/ab_test.py` - A/B testing helpers

In [ ]:
# Reflection Prompt
reflection_questions = [
    "1. Which parser method was most accurate? Why?",
    "2. What threats to validity (prompt leakage, parser bias, imbalance) could distort your conclusions?",
    "3. How many trials would you need to reliably detect a small effect (d=0.2)?",
    "4. If your optimized prompt shows a 10% improvement in priming rate, would you recommend rollout?",
    "5. How might syntactic priming relate to AI safety and alignment?",
]
print("\n".join(reflection_questions))